# Scenario 01: Basic Inference — Temperature and top_p

**QE Perspective:** Sampling parameters directly affect reproducibility and diversity. We validate that:

- **temperature=0.0** yields deterministic results (same prompt → same output across runs).
- **temperature=1.0** allows non-deterministic variation.
- **top_p** (nucleus sampling) is accepted and influences output when used with temperature.

Assume the OGX server is running (e.g. `BASE_URL=http://localhost:8321`). Configuration via `OGX_BASE_URL` / `MODEL_ID` or `BASE_URL` / `MODEL`.


## Setup

Load base URL and model from environment; import `response_text` from `scripts.helpers` (or define inline fallback); create the OGX client.


In [ ]:
import os
from ogx_client import OgxClient
from scripts.helpers import response_text

base_url = os.environ.get("BASE_URL", "http://localhost:8321/v1/")
model = os.environ.get("MODEL")

assert base_url, "OGX_BASE_URL or BASE_URL must be set"
assert model, "MODEL_ID or MODEL must be set"

## Deterministic behavior (temperature=0.0)

With temperature 0, the model should return the same output for the same prompt when called twice.


In [ ]:
client = OgxClient(base_url=base_url)

prompt = "Reply with exactly one number: 42."
r1 = client.responses.create(model=model, input=prompt, temperature=0.0)
r2 = client.responses.create(model=model, input=prompt, temperature=0.0)

assert r1.status == "completed", f"Expected status completed, got {r1.status}"
assert r2.status == "completed", f"Expected status completed, got {r2.status}"
text1 = response_text(r1)
text2 = response_text(r2)
assert text1 and text2, "Expected non-empty response text"
assert text1.strip() == text2.strip(), (
    f"Determinism failed: temp=0.0 should give identical outputs. Got: {text1!r} vs {text2!r}"
)

## Non-deterministic behavior (temperature=1.0)

With temperature 1.0, we only verify that the API completes and returns valid output; we do not require identical responses across runs.


In [ ]:
r = client.responses.create(
    model=model,
    input="Name one European capital city.",
    temperature=1.0,
)
assert r.status == "completed", f"Expected status completed, got {r.status}"
text = response_text(r)
assert text, "Expected non-empty response for temperature=1.0"

## Sampling parameter (temperature only)

Verify that temperature is accepted and a completed response is returned. Note: `top_p` is not supported in the Responses API `responses.create`; use the Chat Completions API for `top_p`.

In [ ]:
r = client.responses.create(
    model=model,
    input="Say hello in one word.",
    temperature=0.3,
)
assert r.status == "completed", f"Expected status completed, got {r.status}"
assert r.output is not None, "Expected output to be present"

## QE Assertions summary

- Deterministic (temp=0.0): two identical prompts yield identical response text.
- Non-deterministic (temp=1.0): response completes with non-empty output.
- Temperature parameter: request completes successfully with output present.